In [2]:
# env 파일 읽어오기

from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv())

MODEL = 'text-embedding-3-small'    # 1536 차원

In [3]:
from openai import OpenAI

client = OpenAI()

response = client.embeddings.create(
    model=MODEL,
    input="나는 오늘 짜장면을 먹었어.",
)

print(response.data[0].embedding)
print(len(response.data[0].embedding))

[0.02130126953125, -0.01514434814453125, -0.0980224609375, 0.00665283203125, 0.0318603515625, 0.029388427734375, 0.032012939453125, 0.057830810546875, -0.0157318115234375, 0.0005540847778320312, -0.0200653076171875, 0.021453857421875, 0.01065826416015625, 0.0673828125, -0.0255584716796875, -0.033416748046875, 0.01187896728515625, 0.031494140625, 0.006748199462890625, 0.0262298583984375, 0.0281982421875, -0.0133056640625, -0.0054168701171875, 0.0009150505065917969, -0.018890380859375, 0.03814697265625, 0.0005574226379394531, -0.009368896484375, -0.0101470947265625, -0.0187530517578125, -0.00010728836059570312, -0.031341552734375, -0.0269622802734375, -0.0472412109375, 0.0030059814453125, -0.0021533966064453125, -0.025238037109375, 0.019866943359375, 0.00379180908203125, 0.03338623046875, -0.02496337890625, -0.015716552734375, 0.020721435546875, 0.00931549072265625, 0.040191650390625, -0.01922607421875, -0.0052642822265625, 0.035675048828125, 0.00897216796875, -0.03057861328125, -0.01788

#### 벡터 유사도
- 벡터 숫자 집합이 나타내는 항목이 서로 얼마나 유사한가?

In [4]:
import numpy as np
from numpy import dot
import pandas as pd
from numpy.linalg import norm

# dot : 내적
# norm : 벡터의 길이

In [5]:
def cosine_similarity(a,b):
    return dot(a,b)/(norm(a)*norm(b))

In [6]:
vec1 = np.array([0,1,1,1])
vec2 = np.array([1,0,1,1])
vec3 = np.array([2,0,2,2])

print("벡터1과 벡터2 유사도", cosine_similarity(vec1,vec2))
print("벡터2과 벡터3 유사도", cosine_similarity(vec2,vec3))
print("벡터3과 벡터1 유사도", cosine_similarity(vec1,vec3))

벡터1과 벡터2 유사도 0.6666666666666667
벡터2과 벡터3 유사도 1.0000000000000002
벡터3과 벡터1 유사도 0.6666666666666667


In [7]:
data = [
    '저는 배가 고파요',
    '저기 배가 지나가네요',
    '굶어서 허기가 지네요',
    '허기 워기라는 게임이 있는데 재밌어요',
    '스팀에서 재밌는거 해야지',
    '스팀에어프라이어로 연어구이 해 먹을거야'
]

df = pd.DataFrame(data,columns=['text'])
df

,text
0,저는 배가 고파요
1,저기 배가 지나가네요
2,굶어서 허기가 지네요
3,허기 워기라는 게임이 있는데 재밌어요
4,스팀에서 재밌는거 해야지
5,스팀에어프라이어로 연어구이 해 먹을거야


In [8]:
def get_embeddings(text):
    response = client.embeddings.create(
        model=MODEL,
        input=text,
    )

    return response.data[0].embedding

In [9]:
# embedding 컬럼 추가

df['embedding'] = df.apply(lambda row:get_embeddings(row.text), axis=1)

In [10]:
df

,text,embedding
0,저는 배가 고파요,"[0.01741, -0.007843, -0.029068, 0.007622, 0.00..."
1,저기 배가 지나가네요,"[0.03949, 0.000087, -0.014183, 0.003641, -0.02..."
2,굶어서 허기가 지네요,"[0.041534, 0.024826, -0.012794, 0.003227, 0.00..."
3,허기 워기라는 게임이 있는데 재밌어요,"[0.009727, 0.019958, -0.054657, -0.038055, -0...."
4,스팀에서 재밌는거 해야지,"[0.007244, 0.053986, -0.041962, 0.007587, 0.00..."
5,스팀에어프라이어로 연어구이 해 먹을거야,"[0.048645, -0.003414, 0.00002, -0.020737, 0.00..."


In [11]:
# 사용자의 문장과 유사성이 가장 높은 3문장 추출

def return_answer_candidate(df, query):
    # 질문도 벡터 변환
    query_embedding = get_embeddings(query)

    # query 문장과 비교하기 위해 유사도 컬럼 생성
    df['similarity'] = df.embedding.apply(lambda x:cosine_similarity(np.array(x), np.array(query_embedding)))

    # top3 추출
    top_three = df.sort_values("similarity", ascending=False).head(3)
    return top_three

In [12]:
sim_result = return_answer_candidate(df, "배에서 꼬르륵 소리가 나네")
sim_result

,text,embedding,similarity
1,저기 배가 지나가네요,"[0.03949, 0.000087, -0.014183, 0.003641, -0.02...",0.311706
4,스팀에서 재밌는거 해야지,"[0.007244, 0.053986, -0.041962, 0.007587, 0.00...",0.276233
2,굶어서 허기가 지네요,"[0.041534, 0.024826, -0.012794, 0.003227, 0.00...",0.272577


### 아마존 고급 음식 리뷰
- amazon-fine-food-reviews

In [13]:
# csv 로드

food_df = pd.read_csv("./data/fine_food_reviews_1k_fully_translated_korean.csv", index_col=0)
food_df.head(5)



,Time,ProductId,UserId,Score,Summary,Text
0,1351123200,B003XPF9BO,A3R7JR3FMEBXQB,5,이런 식으로 어디서 시작하고 멈추는가,시카고 가족에게 가져 오기 위해 일부를 절약하고 싶었지만 노스 캐롤라이나 가족은 4...
1,1351123200,B003JK537S,A3JBPC3WFUT5ZP,1,조각으로 도착했습니다,상자를 열었을 때 전혀 기뻐하지 않습니다. 대부분의 고리가 조각으로 부러졌습니다.
2,1351123200,B000JMBE7M,AQX1N6A51QOKG,4,그것은 Blanc Mange는 아니지만 나쁘지 않습니다,커스터드가 계란이없는 커스터드인지 확실하지 않지만 이것은 비건 채식 팬케이크 레시피...
3,1351123200,B004AHGBX4,A2UY46X0OSNVUQ,3,이들은 또한 소금과 바다 소금이 아닙니다,나는 당신이 당신이 무엇을 얻는 지 알 수 있고 뼈 나 어두운 고기가 없다는 사실을...
4,1351123200,B001BORBHO,A1AFOYZ9HSM2CZ,5,제품에 만족합니다,내 개는 가려운 피부로 고통 스러웠습니다.음식과 피부 문제는 몇 주 안에 정리되었습니다.


In [14]:
# text + summary 결합 컬럼 생성
food_df['Combined'] = "Title: " + food_df.Summary.str.strip() + ", Content: " + food_df.Text.str.strip()



In [15]:
# Time 컬럼 제거
food_df = food_df.drop(columns=['Time'])
food_df.head(3)

,ProductId,UserId,Score,Summary,Text,Combined
0,B003XPF9BO,A3R7JR3FMEBXQB,5,이런 식으로 어디서 시작하고 멈추는가,시카고 가족에게 가져 오기 위해 일부를 절약하고 싶었지만 노스 캐롤라이나 가족은 4...,"Title: 이런 식으로 어디서 시작하고 멈추는가, Content: 시카고 가족에게..."
1,B003JK537S,A3JBPC3WFUT5ZP,1,조각으로 도착했습니다,상자를 열었을 때 전혀 기뻐하지 않습니다. 대부분의 고리가 조각으로 부러졌습니다.,"Title: 조각으로 도착했습니다, Content: 상자를 열었을 때 전혀 기뻐하지..."
2,B000JMBE7M,AQX1N6A51QOKG,4,그것은 Blanc Mange는 아니지만 나쁘지 않습니다,커스터드가 계란이없는 커스터드인지 확실하지 않지만 이것은 비건 채식 팬케이크 레시피...,"Title: 그것은 Blanc Mange는 아니지만 나쁘지 않습니다, Content..."


In [16]:
food_df.shape

(999, 6)

In [21]:
# 임베딩 값 추출
food_df['embedding'] = food_df.Combined.apply(lambda t:get_embeddings(t))
food_df.head(2)

,ProductId,UserId,Score,Summary,Text,Combined,embedding
0,B003XPF9BO,A3R7JR3FMEBXQB,5,이런 식으로 어디서 시작하고 멈추는가,시카고 가족에게 가져 오기 위해 일부를 절약하고 싶었지만 노스 캐롤라이나 가족은 4...,"Title: 이런 식으로 어디서 시작하고 멈추는가, Content: 시카고 가족에게...","[-0.004848, 0.003757, -0.018097, 0.035645, 0.0..."
1,B003JK537S,A3JBPC3WFUT5ZP,1,조각으로 도착했습니다,상자를 열었을 때 전혀 기뻐하지 않습니다. 대부분의 고리가 조각으로 부러졌습니다.,"Title: 조각으로 도착했습니다, Content: 상자를 열었을 때 전혀 기뻐하지...","[0.056519, 0.006905, -0.049774, 0.000397, 0.04..."


In [19]:
food_df.to_csv("./data/food_df.csv")

In [ ]:
from openai import OpenAI
import pandas as pd


df = pd.read_csv("data/food_df.csv", index_col=0)

df['embedding'] = df.embedding.apply(eval).to_list()

df.head(3)

In [ ]:
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv())

MODEL = 'text-embedding-3-small'    # 1536 차원

def cosine_similarity(a,b):
    return dot(a,b)/(norm(a)*norm(b))

def get_embeddings(text):
    response = client.embeddings.create(
        model=MODEL,
        input=text,
    )

    return response.data[0].embedding

def search_reviews(df, query, n=3):
    # 질문도 벡터 변환
    query_embedding = get_embeddings(query)

    # query 문장과 비교하기 위해 유사도 컬럼 생성
    df['similarity'] = df.embedding.apply(lambda x:cosine_similarity(np.array(x), np.array(query_embedding)))

    res = df.sort_values("similarity", ascending=False).head(n)
    return res

In [ ]:
# 유사한 리뷰 찾기
# 질문 => 임베딩

client = OpenAI()

search_reviews(df, '맛있는 콩', 4)



,ProductId,UserId,Score,Summary,Text,Combined,embedding,similarity
771,B002GWHBU2,A2NUQSB42RKBFA,5,자메이카 푸른 콩,로스팅하기위한 우수한 커피 콩 우리 가족은 더 많은 양의 맛을 위해 5 파운드를 더...,"Title: 자메이카 푸른 콩, Content: 로스팅하기위한 우수한 커피 콩 우리...","[0.003361, 0.044891, -0.050995, -0.017578, 0.0...",0.516398
88,B000E3ZFDU,A1PQDL14230X6U,5,맛있는,나는이 흰 콩 조미료를 즐긴다 그것은 콩에 풍부한 맛을줍니다. 나는 법에있는 어머니...,"Title: 맛있는, Content: 나는이 흰 콩 조미료를 즐긴다 그것은 콩에 풍...","[0.046051, 0.034424, -0.074402, -0.007065, 0.0...",0.505980
460,B000RZNY5G,A2ZGTXBI8C7FYI,5,아주 맛있는,아주 좋은 것은 여전히 ​​약간 왼쪽을 가지고 있습니다. 나는 짠 맛이 마음에 든다.,"Title: 아주 맛있는, Content: 아주 좋은 것은 여전히 ​​약간 왼쪽을 ...","[-0.009407, -0.029327, -0.064209, -0.001598, 0...",0.477666
60,B000LVM4EI,ATTK7J5JWVDYD,5,아주 맛있는,최고의 matzo balls 거기에 너무 맛있는 소금으로 너무 맛있어 너무 나쁘다.,"Title: 아주 맛있는, Content: 최고의 matzo balls 거기에 너무...","[0.013397, 0.008118, -0.060059, 0.020172, 0.02...",0.467013


#### FAISS
- Facebook AI Similarity Search
- 벡터 검색 라이브러리 (대량의 벡터 데이터에서 유사한 데이터를 빠르고 효율적으로 겸색하기 위한 도구)
- 인덱스 검색 
    - IndexFlatL2 : 모든 벡터를 비교하는 검색
    - IndexFlatIP : 내적을 이용한 검색

In [ ]:
# uv add faiss-cpu

df.head(2)

,ProductId,UserId,Score,Summary,Text,Combined,embedding,similarity
0,B003XPF9BO,A3R7JR3FMEBXQB,5,이런 식으로 어디서 시작하고 멈추는가,시카고 가족에게 가져 오기 위해 일부를 절약하고 싶었지만 노스 캐롤라이나 가족은 4...,"Title: 이런 식으로 어디서 시작하고 멈추는가, Content: 시카고 가족에게...","[-0.004848, 0.003757, -0.018097, 0.035645, 0.0...",0.229053
1,B003JK537S,A3JBPC3WFUT5ZP,1,조각으로 도착했습니다,상자를 열었을 때 전혀 기뻐하지 않습니다. 대부분의 고리가 조각으로 부러졌습니다.,"Title: 조각으로 도착했습니다, Content: 상자를 열었을 때 전혀 기뻐하지...","[0.056519, 0.006905, -0.049774, 0.000397, 0.04...",0.204593


In [ ]:
# 임베딩을 하나의 행렬로 만들기

print("리뷰개수",len(df))
print(len(df['embedding'].iloc[0]))

vectors = np.array(df['embedding'].tolist(), dtype="float32")
vectors.shape

리뷰개수 999
1536


(999, 1536)

In [ ]:
# 정규화

def to_unit_length(v):
    """벡터 길이 1로 만들기"""
    length = np.linalg.norm(v, axis=1, keepdims=True)
    return v / length

vectors = to_unit_length(vectors)

In [ ]:
import faiss

dim = vectors.shape[1]
index = faiss.IndexFlatIP(dim)    # 내적이용
index.add(vectors)

print("인덱스에 저장된 벡터 수", index.ntotal)

인덱스에 저장된 벡터 수 999


In [ ]:
query_embedding = get_embeddings("맛있는 콩")
query_vec = np.array([query_embedding], dtype="float32")
query_vec = to_unit_length(query_vec)

print(query_vec.shape)
query_vec.head(2)

In [ ]:
scores, positions =  index.search(query_vec, 4)
print("유사도 점수", scores[0])
print("찾은 인덱스", positions[0])

유사도 점수 [0.516398  0.5059799 0.4776663 0.4670134]
찾은 인덱스 [770  88 460  60]


In [ ]:
result = df.iloc[positions[0]].copy()

result['sim'] = scores[0]
result[['Summary', 'Text', 'sim']]

,Summary,Text,sim
771,자메이카 푸른 콩,로스팅하기위한 우수한 커피 콩 우리 가족은 더 많은 양의 맛을 위해 5 파운드를 더...,0.516398
88,맛있는,나는이 흰 콩 조미료를 즐긴다 그것은 콩에 풍부한 맛을줍니다. 나는 법에있는 어머니...,0.505980
460,아주 맛있는,아주 좋은 것은 여전히 ​​약간 왼쪽을 가지고 있습니다. 나는 짠 맛이 마음에 든다.,0.477666
60,아주 맛있는,최고의 matzo balls 거기에 너무 맛있는 소금으로 너무 맛있어 너무 나쁘다.,0.467013
